# Blinkit On-Demand: Bid Push (independent implement script)

Reads ONLY `voylla.blinkit_ondemand_actions` — never `Blinkit_actions_llm`.
Completely separate from `Blinkit_Report_Script_bid_implement.ipynb`; running
this touches nothing that script owns, and running that script touches
nothing this one owns.

**`DRY_RUN = True` by default.** In dry-run, this notebook does everything
except the actual network call to Blinkit: it finds what's due, computes the
exact new CPM for every keyword in the affected campaigns, and prints the
full payload it *would* send. Nothing is marked pushed and nothing changes
on Blinkit. Read the dry-run output for at least one real cycle before
setting `DRY_RUN = False` in the cell below — the same caution you already
apply to Auto Mode on the scheduled side.

**What counts as "due to push":** for each (campaign_id, targeting), only the
MOST RECENT row in `blinkit_ondemand_actions` is considered — an older
accepted row for the same keyword is superseded the moment a newer
suggestion exists for it, accepted or not. A row is due when that latest row
has `user_implemented = 'true'`, its action is `INCREASE_CPM`, `DECREASE_CPM`,
or `PAUSE` (ZOMBIE_FLAG/NO_CHANGE are never pushed), and it hasn't already
been pushed (`push_status` is not `'done'`).

**Payload shape:** Blinkit's API takes a campaign's FULL keyword list per
request, not a single-keyword patch - so every push rebuilds the whole list
from `Blinkit_CPM` (today's live bid sheet), then overwrites only the
keywords that are actually due, exactly like
`Blinkit_Report_Script_bid_implement.ipynb` does it for its own table.

**PAUSE is handled by omission, not a separate API call.** Blinkit has no
per-keyword pause endpoint - never did, in this notebook or the legacy one.
Since the API only accepts a campaign's full keyword list, a keyword simply
left out of that list is exactly as paused as one Blinkit's own dashboard
would show off. Every push re-derives the campaign's FULL currently-paused
set (every keyword whose latest accepted decision is PAUSE, not just ones
freshly due this cycle) and drops them all from the outgoing list, so a
keyword paused last week can't silently reappear just because this week's
push is for a different keyword in the same campaign.

In [ ]:
# The engine code comes from this repo's webapp/ folder - the same code the
# dashboard runs and the tests check. The scheduled job pulls the latest
# main before every run, so a merged change is used by the next run.
# (Credentials still live outside git, in Python_Scripts - db.py reads them there.)
import os
import sys

WEBAPP_DIR = os.environ.get("AI_MARKETING_WEBAPP") or os.path.abspath(os.path.join(os.getcwd(), "..", "webapp"))
sys.path.insert(0, WEBAPP_DIR)

from db import get_db_config
import psycopg2
import psycopg2.extras
from sqlalchemy import create_engine, text
import pandas as pd
import numpy as np
import requests
import json
import time
import random
import platform
import traceback
import imaplib
import email as email_lib
from bs4 import BeautifulSoup
from datetime import datetime, timedelta
import urllib.parse as urlparse

cfg = get_db_config()
Voylla_config = {
    "host": cfg["host"], "user": cfg["user"], "password": cfg["password"],
    "database": cfg["dbname"], "port": int(cfg["port"]),
}
db_connection_str = (
    f"postgresql+psycopg2://{Voylla_config['user']}:{Voylla_config['password']}"
    f"@{Voylla_config['host']}:{Voylla_config['port']}/{Voylla_config['database']}"
)
engine = create_engine(db_connection_str)
print("DB engine ready.")


In [ ]:
# ── SAFETY SWITCH ─────────────────────────────────────────────────────────
DRY_RUN = False
PUSH_BRANDS = ["Voylla", "Chumbak", "Petcrux"]


In [ ]:
# ── Alert email credentials (same account the other Blinkit notebooks use) ──
FL = {'location': r"C:\Users\Amit Singh\Documents\Python_Scripts"}

with open(r"%s\Automationalert_emailid_pass.txt" % FL['location'], 'r') as f:
    lines = [l.strip() for l in f.readlines()]
    Alert_email, Alert_password, Alert_sender_name = lines[0], lines[1], lines[2]

with open(r"%s\automation_mail_receiver.txt" % FL['location'], 'r') as f:
    receiver_email = [l.strip() for l in f.readlines()]

import yagmail

def send_alert_email(subject_suffix, body_message, brand):
    try:
        with yagmail.SMTP(Alert_email, Alert_password) as yag:
            subject = f"Blinkit_Ondemand_Bid_Push - {subject_suffix} ({brand})"
            yag.send(to=receiver_email, subject=subject, contents=body_message,
                      headers={"From": f"{Alert_sender_name} <{Alert_email}>"})
        print(f"Alert email sent for {brand}")
    except Exception as e:
        print(f"Failed to send email alert: {e}")


In [ ]:
# ── Blinkit login helpers (same approach as bid_implement.ipynb - magic-link
#    email, poll for it, never guess; extract the Firebase token from network
#    logs) ──────────────────────────────────────────────────────────────────
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

def human_typing_advanced(element, text):
    element.clear()
    for i, char in enumerate(text):
        element.send_keys(char)
        if i % random.randint(4, 7) == 0:
            time.sleep(random.uniform(0.2, 0.4))
        else:
            time.sleep(random.uniform(0.06, 0.15))

def extract_firebase_token(driver):
    logs = driver.get_log("performance")
    for entry in logs:
        message = json.loads(entry["message"])["message"]
        if message["method"] == "Network.requestWillBeSent":
            hdrs = message.get("params", {}).get("request", {}).get("headers", {})
            if "firebase_user_token" in hdrs:
                return hdrs["firebase_user_token"]
    return None

def blinkit_login(brand, email_1, password_1, sender, ad_account, attempts=3):
    """Returns (driver, token, cookies_dict) or (driver, None, None) on failure."""
    options = Options()
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_argument("--no-sandbox")
    options.add_argument("--start-maximized")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--disable-gpu")
    options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                          "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/119.0.0.0 Safari/537.36")
    options.set_capability("goog:loggingPrefs", {"performance": "ALL"})
    driver = webdriver.Chrome(options=options)
    driver.execute_cdp_cmd("Network.enable", {})

    SUBJECT_KEYWORD = "Blinkit Brand Central"
    IMAP_SERVER = "imap.gmail.com"
    MAIL_WAIT_SECONDS = 180
    MAIL_POLL_SECONDS = 5

    def _search_criteria():
        since_date = (datetime.now() - timedelta(days=1)).strftime("%d-%b-%Y")
        if brand == 'Chumbak':
            return f'(FROM "{sender}" SUBJECT "{SUBJECT_KEYWORD}" SINCE "{since_date}")'
        return f'(FROM "{sender}" TO "{ad_account}" SUBJECT "{SUBJECT_KEYWORD}" SINCE "{since_date}")'

    def _mail_ids():
        m = imaplib.IMAP4_SSL(IMAP_SERVER)
        try:
            m.login(email_1, password_1)
            m.select("INBOX")
            st, msgs = m.search(None, _search_criteria())
            return list(msgs[0].split()) if st == "OK" else []
        except Exception:
            return []
        finally:
            try: m.logout()
            except Exception: pass

    def _fetch_html(mail_id):
        m = imaplib.IMAP4_SSL(IMAP_SERVER)
        try:
            m.login(email_1, password_1)
            m.select("INBOX")
            st, data = m.fetch(mail_id, "(RFC822)")
            if st != "OK":
                return None
            msg = email_lib.message_from_bytes(data[0][1])
            if msg.is_multipart():
                for part in msg.walk():
                    if part.get_content_type() == "text/html":
                        return part.get_payload(decode=True)
                return None
            return msg.get_payload(decode=True)
        except Exception:
            return None
        finally:
            try: m.logout()
            except Exception: pass

    def extract_real_url(url):
        parsed = urlparse.urlparse(url)
        qs = urlparse.parse_qs(parsed.query)
        for key in ["url", "u", "redirect", "target"]:
            if key in qs:
                return qs[key][0]
        return url

    def _link_from_html(html):
        if not html:
            return None
        if isinstance(html, bytes):
            html = html.decode(errors="ignore")
        soup = BeautifulSoup(html, "html.parser")
        for a in soup.find_all("a", href=True):
            text = " ".join(a.get_text(" ", strip=True).lower().split())
            if "sign in" in text:
                return extract_real_url(a["href"])
        return None

    for attempt in range(1, attempts + 1):
        try:
            driver.get("https://brands.blinkit.com/")
            time.sleep(2)
            driver.get("https://brands.blinkit.com/")
            time.sleep(5)
            wait = WebDriverWait(driver, 20)
            login_btn = WebDriverWait(driver, 10).until(
                EC.element_to_be_clickable((By.XPATH, "//button[contains(., 'Log In')]")))
            login_btn.click()
            time.sleep(2)
            username_field = wait.until(
                EC.visibility_of_element_located((By.CSS_SELECTOR, "input[placeholder='Enter your email']")))
            human_typing_advanced(username_field, ad_account)
            request_link = WebDriverWait(driver, 10).until(
                EC.element_to_be_clickable((By.XPATH, "//span[text()='Request Sign in Link']")))

            ids_before = set(_mail_ids())
            request_link.click()
            print(f"   sign-in link requested ({len(ids_before)} older mail(s) already in inbox)")

            link, waited = None, 0
            deadline = time.time() + MAIL_WAIT_SECONDS
            while time.time() < deadline and not link:
                for mid in reversed(_mail_ids()):
                    if mid in ids_before:
                        break
                    link = _link_from_html(_fetch_html(mid))
                    if link:
                        break
                if link:
                    break
                time.sleep(MAIL_POLL_SECONDS)
                waited += MAIL_POLL_SECONDS
                print(f"   waiting for sign-in mail... {waited}s", flush=True)

            if not link:
                raise Exception(f"Sign-in mail did not arrive within {MAIL_WAIT_SECONDS}s (attempt {attempt}/{attempts})")

            driver.get(link)
            time.sleep(2)
            driver.get("https://brands.blinkit.com/diy/list")
            time.sleep(2)

            cookies_dict = {c['name']: c['value'] for c in driver.get_cookies()}
            token = extract_firebase_token(driver)
            if token:
                print("Firebase token captured")
                return driver, token, cookies_dict
            print(f"Firebase token not found (attempt {attempt}/{attempts})")
            time.sleep(50)
        except Exception as e:
            print(f"Login attempt {attempt} failed: {e}")
            traceback.print_exc()

    return driver, None, None


In [ ]:
# ── Fetch what is due for push, per brand ───────────────────────────────────
def fetch_due_for_push(brand):
    query = """
        WITH latest AS (
            SELECT DISTINCT ON (campaign_id, targeting) *
            FROM voylla.blinkit_ondemand_actions
            WHERE "Brand" = %(brand)s
            ORDER BY campaign_id, targeting, requested_at DESC
        )
        SELECT * FROM latest
        WHERE (
                (user_implemented = 'true' AND action IN ('INCREASE_CPM','DECREASE_CPM','PAUSE'))
                OR
                (user_implemented = 'false' AND override_action IN ('INCREASE_CPM','DECREASE_CPM','PAUSE'))
              )
          AND (push_status IS NULL OR push_status <> 'done')
    """
    return pd.read_sql(query, engine, params={"brand": brand})


def fetch_currently_paused(campaign_id, brand):
    """All keywords whose most recent accepted decision for this campaign is
    PAUSE - not just the ones freshly due this cycle. Blinkit_CPM's own sync
    may not always drop a keyword the instant it's paused, so re-deriving the
    full paused set on every push (one extra query per campaign) is safer
    than only tracking newly-due PAUSE rows - it guarantees a keyword paused
    last week can't silently reappear just because this week's push is for a
    different keyword in the same campaign."""
    query = """
        WITH latest AS (
            SELECT DISTINCT ON (campaign_id, targeting) *
            FROM voylla.blinkit_ondemand_actions
            WHERE "Brand" = %(brand)s AND campaign_id::TEXT = %(campaign_id)s
            ORDER BY campaign_id, targeting, requested_at DESC
        )
        SELECT targeting FROM latest
        WHERE (user_implemented = 'true' AND action = 'PAUSE')
           OR (user_implemented = 'false' AND override_action = 'PAUSE')
    """
    df = pd.read_sql(query, engine, params={"brand": brand, "campaign_id": str(campaign_id)})
    return set(df["targeting"])


def effective_action(row):
    """The action that actually applies to this row - the human's override
    if there is one, else the LLM's own action."""
    return row["override_action"] if row["user_implemented"] == "false" else row["action"]


def compute_new_cpm(row):
    """Same math bid_implement.ipynb uses for a human accept/override, with
    the same floor clamp - never below the keyword's floor, never above
    current_cpm on a decrease. Only meaningful for INCREASE_CPM/DECREASE_CPM;
    PAUSE rows never reach this - they're handled by omission instead, see
    fetch_currently_paused / push_campaign."""
    current_cpm = float(row["current_cpm"] or 0)
    floor = float(row["cpm_floor"]) if row["cpm_floor"] else 200.0
    is_override = row["user_implemented"] == "false"
    action = row["override_action"] if is_override else row["action"]
    pct_override = row["cpm_change_user"] if is_override else row["cpm_llm_override"]

    if pct_override is not None:
        pct = float(pct_override)
        raw = current_cpm * (1 + pct / 100) if "INCREASE" in action else current_cpm * (1 - pct / 100)
    else:
        bid_change = float(row["bid_change"] or 0)
        raw = current_cpm + bid_change if "INCREASE" in action else current_cpm - bid_change

    if "DECREASE" in action:
        return int(round(min(current_cpm, max(raw, floor))))
    return int(round(raw))

In [ ]:
# ── Build one full-campaign payload (Blinkit's API takes the whole keyword
#    list, not a single-keyword patch) and push it ─────────────────────────
# Rebuilt to match a REAL captured request from brands.blinkit.com's own
# dashboard (a live PUT while a human edited this exact campaign) instead of
# guessing at the shape - several fields differ from the assumed structure:
#   - dates are "M/D/YYYY" (no leading zeros, no time component), not ISO 8601
#   - campaign_request_type is "UPDATE" for an already-active campaign, not
#     "RESTART" - RESTART is what can revive a campaign that's stopped on
#     purpose; UPDATE is what the real dashboard sends for a bid-only change
#   - keywords are space-separated ("oxidised jhumka"), matching Blinkit_CPM's
#     own convention - blinkit_ondemand_actions stores them underscored for
#     internal consistency, so they need converting back on the way out
#   - products need an "order" field; several top-level fields the real
#     request always sends (even empty) were missing entirely
#
# There is no separate "pause a keyword" call anywhere on Blinkit's API -
# never was, in this notebook or the legacy bid_implement.ipynb it's modeled
# on. Since the API only ever accepts a campaign's FULL keyword list, a
# keyword simply left OUT of that list is exactly as paused as one Blinkit's
# own dashboard would show off. That omission - not a separate endpoint - IS
# the pause mechanism for this whole system.
def fetch_campaign_bid_sheet(campaign_id, brand):
    """Every keyword currently live in this campaign, at its current CPM."""
    return pd.read_sql(
        """
        SELECT REPLACE(cp.keyword, ' ', '_') AS targeting, cp.cpm AS current_cpm
        FROM voylla."Blinkit_CPM" cp
        WHERE cp.campaign_id::TEXT = %(campaign_id)s AND cp."Brand" = %(brand)s
        """,
        engine, params={"campaign_id": str(campaign_id), "brand": brand},
    )


def fetch_campaign_meta(campaign_id):
    df = pd.read_sql(
        """
        SELECT DISTINCT campaign_id, campaign_budget, start_ts, end_ts,
               campaign_type, objective_type, product_id, region_type,
               TRANSLATE(region_ids::TEXT, '{}', '') AS region_ids
        FROM voylla."Blinkit_CampaignWise_ProductID"
        WHERE campaign_id::TEXT = %(campaign_id)s AND region_type IS NOT NULL
        """,
        engine, params={"campaign_id": str(campaign_id)},
    )
    if df.empty:
        return None
    meta = df.iloc[0].to_dict()
    # PRODUCT_LISTING campaigns can have many products - taking only row 0's
    # product_id silently dropped the rest, sending an incomplete/inconsistent
    # payload (this is what caused the first HTTP 500).
    meta["product_ids"] = df["product_id"].dropna().astype(int).unique().tolist()
    return meta


IST_OFFSET = timedelta(hours=5, minutes=30)


def _mdy(dt):
    """M/D/YYYY, no leading zeros, no time - matches the real dashboard's
    request exactly ("8/28/2025", "12/31/9999"). start_ts/end_ts are stored
    in UTC; Blinkit's own dashboard displays and validates against IST, so
    the campaign_start we echo back must be shifted +5:30 too, or the API
    rejects it as "not allowed to be changed" even though it IS the same
    moment - confirmed against a real captured request (2025-08-27 18:30
    UTC in the DB == 8/28/2025 in their UI, not 8/27)."""
    dt = dt + IST_OFFSET
    return f"{dt.month}/{dt.day}/{dt.year}"


def push_campaign(campaign_id, campaign_name, brand, updated_cpm_by_kw, paused_keywords, token, cookies, ad_account):
    """paused_keywords: targeting values to leave entirely out of the
    outgoing keyword list - see the module-level comment above for why
    omission is the actual pause mechanism here."""
    meta = fetch_campaign_meta(campaign_id)
    if meta is None:
        return False, "No campaign metadata found in Blinkit_CampaignWise_ProductID"

    bid_sheet = fetch_campaign_bid_sheet(campaign_id, brand)
    if bid_sheet.empty:
        return False, "No live keywords found in Blinkit_CPM for this campaign"

    bid_sheet = bid_sheet[~bid_sheet["targeting"].isin(paused_keywords)]
    if bid_sheet.empty:
        return False, "Every live keyword in this campaign is paused - nothing left to send"

    result = []
    for _, r in bid_sheet.iterrows():
        kw = r["targeting"]
        cpm = updated_cpm_by_kw.get(kw, int(r["current_cpm"]))
        result.append({
            "keyword": kw.replace("_", " "),
            "bids": [{"match_type": "EXACT", "cpm": int(cpm)}],
        })

    pid_list = meta.get("product_ids") or []

    payload = {
        "source_platform": "diy_dashboard_web",
        "requested_by": "",
        "advertiser_id": 1073,
        "brand_name": "",
        "campaign_id": int(campaign_id),
        "objective_type": meta["objective_type"],
        "asset_type": meta["campaign_type"],
        "image_url": "",
        "header_title": "",
        "creative_type": "",
        "highlighted_pids": "",
        "collection_id": "",
        "store_name": "",
        "name": campaign_name,
        "campaign_start": _mdy(meta["start_ts"]),
        "campaign_end": _mdy(meta["end_ts"]),
        "cpm": 0,
        "campaign_data": {
            "brand_ids": "",
            "category_ids": "",
            "pids": ",".join(map(str, pid_list)),
            "products": [
                {"id": pid, "order": i + 1, "is_star_product": False}
                for i, pid in enumerate(pid_list)
            ],
            "ro_details": {"ro_number": None, "ro_amount": None, "ro_issue_date": None, "proof_url": None},
        },
        "bidding_strategy": {"total_budget": int(meta["campaign_budget"]), "pacing_type": "DAILY"},
        "campaign_targeting": {
            "city_ids": str(meta["region_ids"]),
            "is_extendable": False,
            "days_of_week": [],
            "timeslots": [],
            "keyword_targeting": {"keywords": result},
        },
        "campaign_request_type": "UPDATE",
        "preview_image_url": "",
    }

    if DRY_RUN:
        print(f"  [DRY RUN] would PUT campaign {campaign_id} ({campaign_name}) with "
              f"{len(updated_cpm_by_kw)} changed keyword(s), {len(paused_keywords)} paused (omitted):")
        for kw, cpm in updated_cpm_by_kw.items():
            old = bid_sheet.loc[bid_sheet["targeting"] == kw, "current_cpm"]
            old_val = int(old.iloc[0]) if len(old) else "?"
            print(f"      {kw}: {old_val} -> {cpm}")
        for kw in paused_keywords:
            print(f"      {kw}: PAUSED (left out of the keyword list)")
        return True, "dry_run"

    url = "https://brands.blinkit.com/adservice/v3/campaigns"
    headers = {
        "accept": "application/json, text/plain, */*",
        "content-type": "application/json;charset=UTF-8",
        "origin": "https://brands.blinkit.com",
        "referer": f"https://brands.blinkit.com/diy/edit/campaign/{campaign_id}",
        "user-agent": "Mozilla/5.0",
        "firebase_user_token": token,
    }
    response = requests.put(url, json=payload, headers=headers, cookies=cookies)
    if response.status_code in (200, 201):
        return True, "pushed"
    return False, f"HTTP {response.status_code}: {response.text[:500]}"

In [ ]:
# ── Main loop: one brand at a time, one login per brand, only if that brand
#    has anything due ────────────────────────────────────────────────────
query_brand_login = """SELECT * FROM "DataWarehouse"."Brand_user" a
WHERE a."channel" = 'Email_Pass for Blinkit'"""
df_brand_login = pd.read_sql(query_brand_login, engine)

run_log = []

for brand in PUSH_BRANDS:
    due = fetch_due_for_push(brand)
    if due.empty:
        print(f"{brand}: nothing due for push.")
        continue

    print(f"\n{'='*60}\n{brand}: {len(due)} keyword(s) due for push\n{'='*60}")

    login_row = df_brand_login[df_brand_login["brand"] == brand]
    if login_row.empty:
        print(f"  No Blinkit login on file for {brand} - skipping.")
        continue
    login_row = login_row.iloc[0]

    driver, token, cookies = (None, None, None)
    if not DRY_RUN:
        driver, token, cookies = blinkit_login(
            brand, login_row["email"], login_row["password"], login_row["Sender"], login_row["ad_account_id"]
        )
        if not token:
            print(f"  Could not log in to Blinkit for {brand} - skipping push, nothing marked pushed.")
            if driver:
                driver.quit()
            continue

    for campaign_id, group in due.groupby("campaign_id"):
        campaign_name = group["campaign_name"].iloc[0]
        group = group.copy()
        group["_effective_action"] = group.apply(effective_action, axis=1)

        cpm_rows = group[group["_effective_action"].isin(["INCREASE_CPM", "DECREASE_CPM"])]
        updated_cpm_by_kw = {row["targeting"]: compute_new_cpm(row) for _, row in cpm_rows.iterrows()}

        # Full currently-paused set for this campaign, not just today's newly
        # due PAUSE rows - see fetch_currently_paused's docstring for why.
        paused_keywords = fetch_currently_paused(campaign_id, brand)

        ok, note = push_campaign(
            campaign_id, campaign_name, brand, updated_cpm_by_kw, paused_keywords,
            token, cookies, login_row["ad_account_id"] if not DRY_RUN else None,
        )
        run_log.append({"brand": brand, "campaign_id": campaign_id, "ok": ok, "note": note})
        print(f"  campaign {campaign_id} ({campaign_name}): {'OK' if ok else 'FAILED'} - {note}")

        if not DRY_RUN:
            new_status = "done" if ok else "failed"
            for key in group["unique_key"]:
                with engine.begin() as conn:
                    conn.execute(
                        text("""UPDATE voylla.blinkit_ondemand_actions
                                 SET push_status = :status, pushed_at = NOW(), push_note = :note
                                 WHERE unique_key = :key"""),
                        {"status": new_status, "note": note, "key": key},
                    )
            if ok:
                lines = [f"  {kw}: -> {cpm}" for kw, cpm in updated_cpm_by_kw.items()]
                lines += [f"  {kw}: PAUSED" for kw in group.loc[group["_effective_action"] == "PAUSE", "targeting"]]
                send_alert_email("Push succeeded", f"Campaign {campaign_id} ({campaign_name}) updated:\n" +
                                  "\n".join(lines), brand)
            else:
                send_alert_email("Push FAILED", f"Campaign {campaign_id} ({campaign_name}): {note}", brand)

    if driver:
        driver.quit()

print("\nDone.")
for r in run_log:
    print(r)